# Landmarks processing
Feed the 21 hand landmarks into a MLP.

In [13]:
%pip install datasets mediapipe pillow tqdm

Note: you may need to restart the kernel to use updated packages.


In [14]:
import io
import sys
from pathlib import Path

import numpy as np
from datasets import load_dataset
from PIL import Image
from tqdm.auto import tqdm

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from hand_preprocess import MediaPipeHandPreprocessor

LABEL_NAMES = [
    "call", "dislike", "fist", "four", "like", "mute",
    "no_gesture", "ok", "one", "palm", "peace", "peace_inverted",
    "rock", "stop", "stop_inverted", "three", "three2", "two_up", "two_up_inverted",
]

ds = load_dataset(
    "parquet",
    data_files=str(PROJECT_DIR / "hagrid_small/data/*.parquet"),
    split="train",
)
print(f"Dataset: {len(ds)} samples, {len(LABEL_NAMES)} classes")

Dataset: 153735 samples, 19 classes


In [15]:
CACHE_LANDMARKS = PROJECT_DIR / "landmarks.npy"
CACHE_LABELS = PROJECT_DIR / "labels.npy"

if CACHE_LANDMARKS.exists() and CACHE_LABELS.exists():
    landmarks_arr = np.load(CACHE_LANDMARKS)
    labels_arr = np.load(CACHE_LABELS)
    print(f"Loaded cache: {landmarks_arr.shape}, {labels_arr.shape}")
else:
    landmarks_list, labels_list = [], []
    skipped = 0

    with MediaPipeHandPreprocessor() as preprocessor:
        for sample in tqdm(ds, desc="Extracting landmarks"):
            img = Image.open(io.BytesIO(sample["image"]["bytes"]))
            processed = preprocessor.preprocess_image(img)
            if processed is None:
                skipped += 1
                continue

            _, lm = processed
            landmarks_list.append(lm)
            labels_list.append(sample["label"])

    landmarks_arr = np.stack(landmarks_list)
    labels_arr = np.array(labels_list, dtype=np.int64)
    np.save(CACHE_LANDMARKS, landmarks_arr)
    np.save(CACHE_LABELS, labels_arr)
    print(f"Extracted: {len(landmarks_list)}, skipped: {skipped}")

print(f"landmarks: {landmarks_arr.shape}, labels: {labels_arr.shape}")

Loaded cache: (140938, 21, 2), (140938,)
landmarks: (140938, 21, 2), labels: (140938,)


In [16]:
# Sanity check
print(f"Sample [0] label: {labels_arr[0]} ({LABEL_NAMES[labels_arr[0]]})")
print(f"Sample [0] landmarks:\n{landmarks_arr[0]}")

# Class distribution
unique, counts = np.unique(labels_arr, return_counts=True)
print("\nClass distribution:")
for u, c in zip(unique, counts):
    print(f"  {LABEL_NAMES[u]:20s}: {c}")


Sample [0] label: 0 (call)
Sample [0] landmarks:
[[0.18807636 0.7337178 ]
 [0.23906969 0.57664514]
 [0.3308743  0.42987898]
 [0.40299892 0.3186583 ]
 [0.41547906 0.21183693]
 [0.5043134  0.41497844]
 [0.66777486 0.54172796]
 [0.60786843 0.59740627]
 [0.55086935 0.59027225]
 [0.5017901  0.49564436]
 [0.65621984 0.6182159 ]
 [0.58768743 0.66491675]
 [0.52755004 0.65317816]
 [0.49233082 0.59610945]
 [0.6369965  0.70072186]
 [0.57630247 0.738757  ]
 [0.5171907  0.7295482 ]
 [0.47944775 0.70770156]
 [0.63530874 0.74126756]
 [0.72639716 0.7695559 ]
 [0.80889237 0.78607285]]

Class distribution:
  call                : 6544
  dislike             : 6436
  fist                : 6314
  four                : 6919
  like                : 6381
  mute                : 6402
  no_gesture          : 23713
  ok                  : 6665
  one                 : 6690
  palm                : 6878
  peace               : 6324
  peace_inverted      : 6038
  rock                : 6471
  stop                : 65

In [11]:
import torch
import torch.nn as nn

class LandmarkMLP(nn.Module):
    def __init__(self, input_dim=42, hidden_dim=64, output_dim=128):
        super(LandmarkMLP, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            # randomly zero out 20% of the elements in the input tensor to prevent from overfitting
            nn.Dropout(0.2), 
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim) # output 128-dimensional feature vector
        )

    def forward(self, x):
        # shape of x should be (batch_size, input_dim)
        return self.mlp(x)



In [12]:
model_1b = LandmarkMLP(input_dim=42, hidden_dim=64, output_dim=128)
model_1b.eval() # 切換到評估模式，避免 BatchNorm 在 batch_size = 1 時報錯

tensor_landmarks = torch.tensor(landmarks_arr[0], dtype=torch.float32).view(1, -1)
feature_vector = model_1b(tensor_landmarks)

print("Output feature vector shape:", feature_vector.shape)

Output feature vector shape: torch.Size([1, 128])
